# Geometric Decision Lab — does *one* geometry govern your choices?

You'll make a series of choices about **money gambles**, **splitting money with another person**, and **waiting for a larger reward**. Then you'll fit *your own* decision geometry — the tradeoff **angles** a model of choice assigns you — and test a real, open research question: do the *same* people who are more **loss-averse** in risky choice also show more **inequality-aversion** in fairness choices? (i.e., is there one shared decision metric across domains?)

**Voluntary, not for credit.** This is a live pre-registered study (`prereg-coupling-angle-v1`); the honest answer is unknown, and volunteers help decide it.

## Consent (read first)

- Participation is **entirely voluntary and not for credit** — no grade is attached, and choosing not to take part, or stopping at any point, has **no consequence** whatsoever.
- Your results are **anonymous**: no name or identity is stored — only your fitted angles, and only if you opt in (`CONTRIBUTE = True` at the end).
- You can do the whole notebook just to see *your own* decision geometry and share nothing.
- *Instructor: this cell is a placeholder for your IRB protocol number and approved consent language before any data is collected.*

In [ ]:
import numpy as np, json, hashlib
try:
    import ipywidgets as W
    from IPython.display import display
    HAVE_WIDGETS = True
except Exception:
    HAVE_WIDGETS = False

# ---- the frozen choice menus (embedded; each: [text_A, text_B, encoded_A(d1,d2), encoded_B(d1,d2)]) ----
MENUS = {"risk_gain": [["50% chance $16.50, 50% chance $15.50", "50% chance $23.00, 50% chance $16.00", [0.4, 0.05], [0.05, 0.35]], ["50% chance $15.50, 50% chance $13.50", "50% chance $23.50, 50% chance $14.00", [0.55, 0.1], [0.125, 0.475]], ["50% chance $15.50, 50% chance $11.00", "50% chance $25.00, 50% chance $10.50", [0.675, 0.225], [0.225, 0.725]], ["50% chance $17.50, 50% chance $13.00", "50% chance $23.00, 50% chance $12.50", [0.475, 0.225], [0.225, 0.525]], ["50% chance $17.50, 50% chance $13.50", "50% chance $22.00, 50% chance $13.00", [0.45, 0.2], [0.25, 0.45]], ["50% chance $19.00, 50% chance $12.00", "50% chance $21.50, 50% chance $11.50", [0.45, 0.35], [0.35, 0.5]], ["50% chance $18.50, 50% chance $11.00", "50% chance $20.50, 50% chance $11.00", [0.525, 0.375], [0.425, 0.475]], ["50% chance $19.50, 50% chance $14.00", "50% chance $20.00, 50% chance $14.00", [0.325, 0.275], [0.3, 0.3]], ["50% chance $20.00, 50% chance $11.50", "50% chance $19.50, 50% chance $10.50", [0.425, 0.425], [0.5, 0.45]], ["50% chance $21.00, 50% chance $12.50", "50% chance $19.50, 50% chance $12.50", [0.325, 0.425], [0.4, 0.35]], ["50% chance $21.50, 50% chance $11.00", "50% chance $18.00, 50% chance $11.50", [0.375, 0.525], [0.525, 0.325]], ["50% chance $22.00, 50% chance $14.00", "50% chance $18.50, 50% chance $14.50", [0.2, 0.4], [0.35, 0.2]], ["50% chance $22.00, 50% chance $15.00", "50% chance $18.00, 50% chance $15.00", [0.15, 0.35], [0.35, 0.15]], ["50% chance $24.00, 50% chance $11.50", "50% chance $15.50, 50% chance $11.00", [0.225, 0.625], [0.675, 0.225]], ["50% chance $23.50, 50% chance $14.50", "50% chance $17.00, 50% chance $14.50", [0.1, 0.45], [0.425, 0.125]], ["50% chance $26.50, 50% chance $11.00", "50% chance $13.50, 50% chance $11.50", [0.125, 0.775], [0.75, 0.1]]], "risk_loss": [["50% chance lose $15.50, 50% chance lose $14.00", "50% chance lose $24.00, 50% chance lose $14.50", [0.525, 0.075], [0.075, 0.475]], ["50% chance lose $15.00, 50% chance lose $12.00", "50% chance lose $24.50, 50% chance lose $12.00", [0.65, 0.15], [0.175, 0.625]], ["50% chance lose $15.50, 50% chance lose $11.50", "50% chance lose $25.00, 50% chance lose $11.00", [0.65, 0.2], [0.2, 0.7]], ["50% chance lose $17.00, 50% chance lose $11.50", "50% chance lose $23.00, 50% chance lose $11.50", [0.575, 0.275], [0.275, 0.575]], ["50% chance lose $17.50, 50% chance lose $13.00", "50% chance lose $22.50, 50% chance lose $12.50", [0.475, 0.225], [0.25, 0.5]], ["50% chance lose $18.00, 50% chance lose $12.50", "50% chance lose $21.50, 50% chance lose $13.00", [0.475, 0.275], [0.275, 0.425]], ["50% chance lose $19.50, 50% chance lose $15.00", "50% chance lose $20.50, 50% chance lose $15.00", [0.275, 0.225], [0.225, 0.275]], ["50% chance lose $20.00, 50% chance lose $13.50", "50% chance lose $20.50, 50% chance lose $13.00", [0.325, 0.325], [0.325, 0.375]], ["50% chance lose $20.50, 50% chance lose $11.00", "50% chance lose $20.00, 50% chance lose $10.00", [0.425, 0.475], [0.5, 0.5]], ["50% chance lose $21.00, 50% chance lose $12.00", "50% chance lose $18.50, 50% chance lose $12.00", [0.35, 0.45], [0.475, 0.325]], ["50% chance lose $21.50, 50% chance lose $14.00", "50% chance lose $19.00, 50% chance lose $14.00", [0.225, 0.375], [0.35, 0.25]], ["50% chance lose $21.50, 50% chance lose $14.00", "50% chance lose $18.00, 50% chance lose $14.00", [0.225, 0.375], [0.4, 0.2]], ["50% chance lose $21.50, 50% chance lose $15.50", "50% chance lose $18.00, 50% chance lose $15.50", [0.15, 0.3], [0.325, 0.125]], ["50% chance lose $24.00, 50% chance lose $11.50", "50% chance lose $15.00, 50% chance lose $10.50", [0.225, 0.625], [0.725, 0.225]], ["50% chance lose $25.00, 50% chance lose $11.50", "50% chance lose $14.50, 50% chance lose $10.50", [0.175, 0.675], [0.75, 0.2]], ["50% chance lose $27.50, 50% chance lose $10.50", "50% chance lose $12.50, 50% chance lose $10.00", [0.1, 0.85], [0.875, 0.125]]], "social_adv": [["you get $19.50, other gets $14.00", "you get $24.00, other gets $9.00", [0.55, 0.1], [0.1, 0.6]], ["you get $18.50, other gets $13.50", "you get $23.50, other gets $9.00", [0.65, 0.15], [0.15, 0.6]], ["you get $17.50, other gets $12.50", "you get $22.50, other gets $8.00", [0.75, 0.25], [0.25, 0.7]], ["you get $18.00, other gets $11.50", "you get $22.00, other gets $7.50", [0.7, 0.35], [0.3, 0.75]], ["you get $19.00, other gets $12.00", "you get $21.50, other gets $8.50", [0.6, 0.3], [0.35, 0.65]], ["you get $21.00, other gets $12.00", "you get $22.00, other gets $11.00", [0.4, 0.3], [0.3, 0.4]], ["you get $20.50, other gets $11.50", "you get $21.50, other gets $10.50", [0.45, 0.35], [0.35, 0.45]], ["you get $21.00, other gets $10.50", "you get $20.50, other gets $10.50", [0.4, 0.45], [0.45, 0.45]], ["you get $22.50, other gets $12.00", "you get $22.50, other gets $12.50", [0.25, 0.3], [0.25, 0.25]], ["you get $22.00, other gets $11.50", "you get $21.00, other gets $12.00", [0.3, 0.35], [0.4, 0.3]], ["you get $23.00, other gets $12.00", "you get $22.00, other gets $13.00", [0.2, 0.3], [0.3, 0.2]], ["you get $21.50, other gets $9.50", "you get $19.00, other gets $12.00", [0.35, 0.55], [0.6, 0.3]], ["you get $23.00, other gets $10.50", "you get $20.00, other gets $13.00", [0.2, 0.45], [0.5, 0.2]], ["you get $23.00, other gets $9.50", "you get $19.00, other gets $13.00", [0.2, 0.55], [0.6, 0.2]], ["you get $24.00, other gets $11.50", "you get $21.00, other gets $14.00", [0.1, 0.35], [0.4, 0.1]], ["you get $23.50, other gets $8.00", "you get $17.00, other gets $13.50", [0.15, 0.7], [0.8, 0.15]]], "social_dis": [["you get $11.00, other gets $24.50", "you get $14.50, other gets $20.50", [0.4, 0.05], [0.05, 0.45]], ["you get $7.00, other gets $23.50", "you get $13.00, other gets $17.00", [0.8, 0.15], [0.2, 0.8]], ["you get $8.00, other gets $23.00", "you get $13.00, other gets $18.50", [0.7, 0.2], [0.2, 0.65]], ["you get $11.00, other gets $23.00", "you get $13.50, other gets $21.50", [0.4, 0.2], [0.15, 0.35]], ["you get $12.00, other gets $23.00", "you get $13.50, other gets $22.00", [0.3, 0.2], [0.15, 0.3]], ["you get $12.00, other gets $23.00", "you get $13.00, other gets $22.50", [0.3, 0.2], [0.2, 0.25]], ["you get $9.50, other gets $20.50", "you get $11.00, other gets $19.50", [0.55, 0.45], [0.4, 0.55]], ["you get $12.50, other gets $22.50", "you get $13.00, other gets $22.50", [0.25, 0.25], [0.2, 0.25]], ["you get $11.00, other gets $21.00", "you get $11.00, other gets $21.00", [0.4, 0.4], [0.4, 0.4]], ["you get $13.00, other gets $22.50", "you get $12.50, other gets $23.00", [0.2, 0.25], [0.25, 0.2]], ["you get $11.50, other gets $19.50", "you get $10.00, other gets $21.50", [0.35, 0.55], [0.5, 0.35]], ["you get $13.00, other gets $21.00", "you get $11.50, other gets $23.00", [0.2, 0.4], [0.35, 0.2]], ["you get $13.00, other gets $20.50", "you get $10.00, other gets $23.00", [0.2, 0.45], [0.5, 0.2]], ["you get $13.00, other gets $19.50", "you get $9.50, other gets $23.00", [0.2, 0.55], [0.55, 0.2]], ["you get $14.00, other gets $21.50", "you get $11.50, other gets $24.00", [0.1, 0.35], [0.35, 0.1]], ["you get $14.50, other gets $19.00", "you get $9.50, other gets $24.00", [0.05, 0.6], [0.55, 0.1]]], "patience": [["$18.50 in 6 days", "$24.00 in 37 days", [0.65, 0.1], [0.1, 0.6167]], ["$17.50 in 12 days", "$23.00 in 49 days", [0.75, 0.2], [0.2, 0.8167]], ["$18.50 in 14 days", "$22.50 in 41 days", [0.65, 0.2333], [0.25, 0.6833]], ["$18.50 in 16 days", "$22.00 in 44 days", [0.65, 0.2667], [0.3, 0.7333]], ["$18.00 in 22 days", "$21.50 in 40 days", [0.7, 0.3667], [0.35, 0.6667]], ["$21.50 in 16 days", "$22.50 in 22 days", [0.35, 0.2667], [0.25, 0.3667]], ["$21.50 in 14 days", "$22.50 in 19 days", [0.35, 0.2333], [0.25, 0.3167]], ["$21.00 in 21 days", "$21.50 in 24 days", [0.4, 0.35], [0.35, 0.4]], ["$20.50 in 30 days", "$20.00 in 27 days", [0.45, 0.5], [0.5, 0.45]], ["$21.50 in 26 days", "$21.00 in 19 days", [0.35, 0.4333], [0.4, 0.3167]], ["$22.00 in 28 days", "$20.00 in 19 days", [0.3, 0.4667], [0.5, 0.3167]], ["$22.00 in 29 days", "$19.50 in 17 days", [0.3, 0.4833], [0.55, 0.2833]], ["$23.50 in 22 days", "$21.50 in 9 days", [0.15, 0.3667], [0.35, 0.15]], ["$24.00 in 20 days", "$21.50 in 6 days", [0.1, 0.3333], [0.35, 0.1]], ["$23.50 in 37 days", "$18.00 in 9 days", [0.15, 0.6167], [0.7, 0.15]], ["$23.50 in 51 days", "$16.50 in 8 days", [0.15, 0.85], [0.85, 0.1333]]]}
BLOCKS = ["risk_gain","risk_loss","social_adv","social_dis","patience"]
T = 0.20                                   # choice temperature (fixed, from the theory)
GRID = np.linspace(0.05, np.pi/2-0.05, 61) # candidate tradeoff angles

def _cost(theta, d):                       # geometric cost of an option with shortfalls d=(d1,d2)
    return np.cos(theta)*d[0] + np.sin(theta)*d[1]

def fit_angle(choices, menu):
    """Grid-MLE of one person's tradeoff angle from their A/B choices in a block.
    choices: list of 1(=A)/0(=B); menu: the block's rows. Returns the best-fit angle (radians)."""
    y = np.array(choices, float)
    best_ll, best = -1e9, GRID[len(GRID)//2]
    for th in GRID:
        pA = 1/(1+np.exp(-(np.array([_cost(th,r[3])-_cost(th,r[2]) for r in menu]))/T))
        pA = np.clip(pA, 1e-6, 1-1e-6)
        ll = float(np.sum(y*np.log(pA)+(1-y)*np.log(1-pA)))
        if ll > best_ll: best_ll, best = ll, th
    return float(best)
print("setup ok — widgets:", HAVE_WIDGETS)

## Part 1 — Make your choices
Imagine each amount is real. Pick the option you'd truly take.

In [ ]:
# Make your choices. For each row pick the option you would actually prefer (real money framing).
# If ipywidgets renders below, click A/B and then run the next cell. Otherwise, edit `responses`
# manually (1 = you prefer A, 0 = you prefer B), one list per block.
responses = {}
_widgets = {}
if HAVE_WIDGETS:
    boxes = []
    for b in BLOCKS:
        rows = []
        for i,(ta,tb,_,_) in enumerate(MENUS[b]):
            w = W.ToggleButtons(options=[("A: "+ta,1),("B: "+tb,0)], value=1,
                                style={"button_width":"initial"})
            _widgets.setdefault(b,[]).append(w); rows.append(w)
        boxes.append(W.VBox([W.HTML(f"<b>{b}</b>")]+rows))
    display(W.VBox(boxes))
    print("Pick your choices above, then run the next cell.")
else:
    # fallback: a placeholder set of choices so the notebook runs end-to-end. REPLACE with your own.
    rng = np.random.default_rng(0)
    responses = {b: list(rng.integers(0,2,len(MENUS[b]))) for b in BLOCKS}
    print("ipywidgets not available — using placeholder choices. Edit `responses` with your real picks.")

In [ ]:
# collect the widget choices (run after picking above)
if HAVE_WIDGETS and _widgets:
    responses = {b: [w.value for w in _widgets[b]] for b in BLOCKS}
print({b: f"{sum(responses[b])}/{len(responses[b])} chose A" for b in BLOCKS})

## Part 2 — Fit your decision geometry
A tradeoff **angle** per block; the **difference** between the two reflection sub-blocks (gain vs loss, advantageous vs disadvantageous) is your **aversion parameter** in that domain.

In [ ]:
# Fit YOUR decision geometry: one tradeoff angle per sub-block, then your aversion parameters.
theta = {b: fit_angle(responses[b], MENUS[b]) for b in BLOCKS}
a_risk   = theta["risk_gain"]  - theta["risk_loss"]   # your loss-aversion parameter (risk reflection)
a_social = theta["social_adv"] - theta["social_dis"]  # your inequality-aversion parameter (social reflection)
print("your sub-angles (rad):", {k: round(v,3) for k,v in theta.items()})
print(f"your loss-aversion  a_risk   = {a_risk:+.3f}")
print(f"your inequality-av. a_social = {a_social:+.3f}")

## Part 3 — Contribute (optional, anonymous)

In [ ]:
# OPTIONAL, ANONYMOUS research contribution. This writes ONLY your angles + aversion parameters
# (no name, no raw identity). Submitting is voluntary and not part of your grade. To opt in, set
# CONTRIBUTE = True, run, and upload the printed file to the course dropbox.
CONTRIBUTE = False
record = {"a_risk": a_risk, "a_social": a_social,
          "theta": {k: round(v,4) for k,v in theta.items()},
          "id": hashlib.sha256(str(sorted(responses.items())).encode()).hexdigest()[:12]}
if CONTRIBUTE:
    fn = f"coupling_response_{record['id']}.json"
    json.dump(record, open(fn,"w"), indent=2)
    print("wrote", fn, "-> upload this file (anonymous).")
else:
    print("Not contributing (CONTRIBUTE=False). Your record would be:", record)

## Part 4 — The class result (instructor / after collection)
The pre-registered PRIMARY test: correlate every student's loss-aversion with their inequality-aversion. A shared decision metric predicts a **positive** coupling.

In [ ]:
# INSTRUCTOR / class-level analysis. After collecting the anonymous JSONs into ./responses/, this
# runs the pre-registered PRIMARY test: does each student's loss-aversion couple with their
# inequality-aversion? (the aversion contrast, prereg-coupling-angle-v1). Needs ~200 for full power.
import glob
recs = [json.load(open(f)) for f in glob.glob("responses/*.json")]
if len(recs) < 10:
    print(f"only {len(recs)} responses — collect more (target ~200 for full power).")
else:
    ar = np.array([r["a_risk"] for r in recs]); as_ = np.array([r["a_social"] for r in recs])
    r = float(np.corrcoef(ar, as_)[0,1])
    boots = np.array([np.corrcoef(*np.array([[ar[i],as_[i]] for i in
             np.random.default_rng(k).integers(0,len(ar),len(ar))]).T)[0,1] for k in range(2000)])
    lo, hi = np.percentile(boots,[2.5,97.5])
    print(f"N={len(recs)}  aversion coupling r = {r:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")
    print("PASS (Level-B supported at the aversion rank)" if lo>0.10 else
          "not yet decisive (CI includes 0.10) — collect more or report as inconclusive.")

## Optional exercises (to go deeper)
1. Explain why fitting **one angle** per domain and correlating misses the coupling, but the **gain−loss difference** finds it. (Hint: what does averaging over the reflection do to a factor that flips sign across it?)
2. Modify `fit_angle` to return a bootstrap confidence interval for *your* angle.
3. The class `r` has a wide CI at N≈100. How many students are needed for the CI to exclude 0.10? Simulate it.
4. What would it mean for decision theory if the coupling is **zero**? Argue both sides.